In [1]:
import torch
import argparse
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import ConcatDataset
from torchvision import datasets, transforms, models

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
device

'cpu'

In [4]:
def get_transforms(dataset_name, img_size=224):

    """
    
    """
    
    if dataset_name == "mnist":
        return transforms.Compose([ transforms.Resize(224),
                                    transforms.Grayscale(num_output_channels=3),
                                    transforms.ToTensor(),
                                    transforms.Normalize((0.485, 0.456, 0.406),
                                                         (0.229, 0.224, 0.225))
                                ])
    
    elif dataset_name == "cifar10":
        return transforms.Compose([ transforms.Resize(224),
                                    transforms.ToTensor(),
                                    transforms.Normalize((0.485, 0.456, 0.406),
                                                         (0.229, 0.224, 0.225))])
    
    elif dataset_name == "flowers":
        return transforms.Compose([
                                    transforms.Resize((224, 224)),
                                    transforms.ToTensor(),
                                    transforms.Normalize((0.485, 0.456, 0.406),
                                                         (0.229, 0.224, 0.225))
                                ])

In [5]:
def get_dataset(name, transform, combine_dataset=True):
    

    """
    
    """
    
    
    if name == "mnist":
        
        # Load Datasets
        train_dataset = datasets.MNIST(
                                        root="./data",
                                        train=True,
                                        download=True,
                                        transform=transform
                                       )
        
        test_dataset = datasets.MNIST(
                                        root="./data",
                                        train=False,
                                        download=True,
                                        transform=transform
                                      )
        num_classes = 10


    
    
    elif name == "cifar10":
        
        # Load Datasets
        train_dataset = datasets.CIFAR10(root='./data', 
                                         train=True, 
                                         download=True,
                                         transform=transform)
    
        test_dataset = datasets.CIFAR10(root='./data', 
                                        train=False, 
                                        download=True,
                                        transform=transform)
        
        num_classes = 10


    

    elif name == "flowers":
        
        # Load Datasets
        train_dataset = datasets.Flowers102(
                                                root="./data",
                                                split="train",
                                                download=True,
                                                transform=transform
                                            )
    
        val_dataset = datasets.Flowers102(
                                        root="./data",
                                        split="val",
                                        download=True,
                                        transform=transform
                                      )
    
        
        test_dataset = datasets.Flowers102(
                                            root="./data",
                                            split="test",
                                            download=True,
                                            transform=transform
                                          )
    
    
        # Form the final training dataset
        if combine_dataset:
            train_dataset = ConcatDataset([train_dataset, val_dataset])
        else:
            train_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])
            
        num_classes = 102

    
    # Get classes of the test data
    class_names = test_dataset.classes
    
    return train_dataset, test_dataset, num_classes, class_names

In [6]:
def get_model(num_classes):
    
    """
    
    """
    
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    return model

In [7]:
def train_model(model, optimizer, criterion, scheduler, train_loader, test_loader, epochs=100, min_delta=0.0001, patience=10, dataset='mnist', device=device):

    """
    
    """

    
    #
    best_acc = 0
    patience_counter = 0
    

    #
    for epoch in range(epochs):

        model.train()

        total_loss = 0
    
        for images, labels in train_loader:
            
            images, labels = images.to(device), labels.to(device)
    
            optimizer.zero_grad()
    
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
    
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
    
            total_loss += loss.item()

        loss = total_loss / len(train_loader)
        acc = evaluate(model, test_loader)
        scheduler.step()

        print(f"Epoch {epoch+1}: Train Loss={loss:.4f}, Test Acc={acc:.4f}")

        
        # Check improvement
        if acc > best_acc + min_delta:
            
            best_acc = acc
            patience_counter = 0

            torch.save(model.state_dict(), f"{dataset}_best.pth")
            print("New best model saved")
            

        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{patience})")

        # Early stopping condition
        if patience_counter >= patience:
            print(f"\n Early stopping triggered at epoch {epoch+1}")
            break

    print(f"Best Accuracy: {best_acc:.4f}")

In [8]:
def evaluate(model, loader, device=device):
    
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        
        for images, labels in loader:
            
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [9]:
datasets_options = ["mnist", "cifar10", "flowers"]

In [10]:
chosen_dataset = datasets_options[0]
transform = get_transforms(chosen_dataset)
train_set, test_set, num_classes, class_names = get_dataset(chosen_dataset, transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 444kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.95MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.65MB/s]


In [11]:
print(f"{len(class_names)} classes found with labels:", class_names)

10 classes found with labels: ['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']


In [12]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [13]:
model = get_model(num_classes).to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]


In [14]:
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

In [15]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

In [16]:
criterion = nn.CrossEntropyLoss()

In [17]:
scaler = torch.amp.GradScaler(device=device)

In [ ]:
train_model(model, optimizer, criterion, scheduler, train_loader, test_loader, epochs=100, dataset=chosen_dataset, device=device)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipykernel_55/1610556837.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
